In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [2]:
!pip install numpy pandas matplotlib scikit-learn seaborn

In [3]:
data = pd.read_csv("data/heart.csv")
data.head()

FileNotFoundError: [Errno 2] No such file or directory: 'data/heart.csv'

In [ ]:
data.shape

In [ ]:
data.info()

In [ ]:
data['target'].value_counts()

In [ ]:
X = data.drop("target", axis=1).values
y = data["target"].values

# Convert labels: 0 → -1
y = np.where(y == 0, -1, 1)

print("Feature shape:", X.shape)
print("Label shape:", y.shape)

## Feature Scaling 

In [ ]:
scaler = StandardScaler()
X = scaler.fit_transform(X)

## Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

In [ ]:
class SVM:
    
    def __init__(self, learning_rate=0.0001, lambda_param=0.001, n_iters=3000):
        self.lr = learning_rate
        self.lambda_param = lambda_param
        self.n_iters = n_iters
        self.w = None
        self.b = None
        self.losses = []
        
    def fit(self, X, y):
        n_samples, n_features = X.shape
        
        self.w = np.zeros(n_features)
        self.b = 0
        
        for _ in range(self.n_iters):
            
            indices = np.arange(n_samples)
            np.random.shuffle(indices)
            
            for idx in indices:
                x_i = X[idx]
                y_i = y[idx]
                
                condition = y_i * (np.dot(x_i, self.w) + self.b) >= 1
                
                if condition:
                    self.w -= self.lr * (2 * self.lambda_param * self.w)
                else:
                    self.w -= self.lr * (2 * self.lambda_param * self.w - y_i * x_i)
                    self.b += self.lr * y_i
            
            # Track loss
            loss = 0
            for i in range(n_samples):
                loss += max(0, 1 - y[i] * (np.dot(X[i], self.w) + self.b))
            loss += self.lambda_param * np.dot(self.w, self.w)
            self.losses.append(loss)
    
    def predict(self, X):
        approx = np.dot(X, self.w) + self.b
        return np.sign(approx)

## Train Model

In [ ]:
model = SVM(learning_rate=0.0001, lambda_param=0.001, n_iters=5000)
model.fit(X_train, y_train)

## Make Predictions

In [ ]:
predictions = model.predict(X_test)

## Evaluate Model

In [ ]:
accuracy = accuracy_score(y_test, predictions)

print("Accuracy:", accuracy)
print("\nConfusion Matrix:\n", confusion_matrix(y_test, predictions))
print("\nClassification Report:\n", classification_report(y_test, predictions))

## Plot Loss vs Iteration

In [ ]:
plt.plot(model.losses)
plt.title("Loss vs Iterations")
plt.xlabel("Iteration")
plt.ylabel("Loss")
plt.show()

##  Confusion Matrix Heatmap

In [ ]:
cm = confusion_matrix(y_test, predictions)

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

# Compare With Sklearn (Analysis Only)

In [ ]:
from sklearn.svm import SVC

sk_model = SVC(kernel='rbf')
sk_model.fit(X_train, y_train)

sk_predictions = sk_model.predict(X_test)

print("Sklearn SVM Accuracy:", accuracy_score(y_test, sk_predictions))

#  Hyperparameter Experiment

In [ ]:
lambdas = [0.0001, 0.001, 0.01, 0.1]
accuracies = []

for l in lambdas:
    temp_model = SVM(lambda_param=l)
    temp_model.fit(X_train, y_train)
    preds = temp_model.predict(X_test)
    accuracies.append(accuracy_score(y_test, preds))

plt.plot(lambdas, accuracies)
plt.xlabel("Lambda")
plt.ylabel("Accuracy")
plt.title("Effect of Regularization")
plt.show()

# From-Scratch Linear vs RBF SVM (Project Models)

In this section we:

- Import the final project SVM implementations from `svm.py`:
  - `ScratchLinearSVM` (primal SGD, linear)
  - `KernelSVM` (SMO-style, supports linear / RBF kernels)
- Reuse the same `heart.csv` data and preprocessing.
- Train both models on the same train/test split.
- Plot accuracy / hinge-loss vs iterations (where available) and print a clear side-by-side comparison.

In [4]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# Import the project SVM implementations
from svm import ScratchLinearSVM, KernelSVM

# Reload the dataset to be independent of earlier cells
# Reload the dataset to be independent of earlier cells
BASE_DIR = Path.cwd()
DATA_PATH = BASE_DIR / "heart.csv"


heart_df = pd.read_csv(DATA_PATH)
X_full = heart_df.drop("target", axis=1).values
y_full = heart_df["target"].values.astype(int)

# Train / test split (match project settings)
X_train_svm, X_test_svm, y_train_svm, y_test_svm = train_test_split(
    X_full, y_full, test_size=0.2, random_state=42, stratify=y_full
)

# Standardise features (as in the project script)
scaler_svm = StandardScaler()
X_train_svm_scaled = scaler_svm.fit_transform(X_train_svm)
X_test_svm_scaled = scaler_svm.transform(X_test_svm)

print("Train shape:", X_train_svm_scaled.shape)
print("Test shape:", X_test_svm_scaled.shape)

Train shape: (242, 13)
Test shape: (61, 13)


In [5]:
# Train ScratchLinearSVM (linear) and KernelSVM (RBF)

# 1) Linear SVM (from-scratch, primal SGD)
y_train_linear_internal = np.where(y_train_svm == 0, -1, 1).astype(float)

linear_model = ScratchLinearSVM(
    learning_rate=0.0001,
    lambda_param=0.001,
    n_iters=5000,
)
linear_model.fit(X_train_svm_scaled, y_train_linear_internal)

linear_train_pred = linear_model.predict(X_train_svm_scaled)
linear_test_pred = linear_model.predict(X_test_svm_scaled)

linear_train_acc = accuracy_score(y_train_svm, linear_train_pred) * 100
linear_test_acc = accuracy_score(y_test_svm, linear_test_pred) * 100

print(f"ScratchLinearSVM - Train Accuracy: {linear_train_acc:.2f}%")
print(f"ScratchLinearSVM - Test Accuracy:  {linear_test_acc:.2f}%")



ScratchLinearSVM - Train Accuracy: 85.95%
ScratchLinearSVM - Test Accuracy:  78.69%


In [6]:
# 2) RBF Kernel SVM (from-scratch, SMO-style)
y_train_rbf_internal = np.where(y_train_svm == 0, -1, 1).astype(float)

rbf_model = KernelSVM(
    C=1.0,
    tol=1e-3,
    max_iter=1000,
    kernel="rbf",
    gamma=0.01,
)
rbf_model.fit(X_train_svm_scaled, y_train_rbf_internal)

rbf_train_pred = rbf_model.predict(X_train_svm_scaled)
rbf_test_pred = rbf_model.predict(X_test_svm_scaled)

rbf_train_acc = accuracy_score(y_train_svm, rbf_train_pred) * 100
rbf_test_acc = accuracy_score(y_test_svm, rbf_test_pred) * 100

print(f"KernelSVM (RBF) - Train Accuracy: {rbf_train_acc:.2f}%")
print(f"KernelSVM (RBF) - Test Accuracy:  {rbf_test_acc:.2f}%")

KernelSVM (RBF) - Train Accuracy: 84.71%
KernelSVM (RBF) - Test Accuracy:  81.97%


In [ ]:
# Simple visual comparison of linear vs RBF SVM

models = ["Linear (Scratch)", "RBF (Scratch)"]
train_accs = [linear_train_acc, rbf_train_acc]
test_accs = [linear_test_acc, rbf_test_acc]

x = np.arange(len(models))
width = 0.35

plt.figure(figsize=(8, 5))
plt.bar(x - width / 2, train_accs, width, label="Train")
plt.bar(x + width / 2, test_accs, width, label="Test")
plt.xticks(x, models)
plt.ylabel("Accuracy (%)")
plt.title("From-Scratch Linear vs RBF SVM (Heart Disease)")
plt.ylim(0, 100)
plt.legend()
plt.show()

print("\nSummary:")
for name, tr, te in zip(models, train_accs, test_accs):
    print(f"{name:18s} -> Train: {tr:6.2f}% | Test: {te:6.2f}%")